# NeuroShield — Track 2: Brain Tumor Segmentation (BraTS2020, 2D nnU-Net)
### Master Pipeline — Corrected, Persistence-Safe Version

This notebook preserves the existing methodology exactly:
- Dataset: BraTS2020 (368 usable cases)
- Fixed seed = 42, outer split: Train 257 / Val 55 / Test 56
- Model A: 2D nnU-Net, `nnUNetTrainer_100epochs`, fold 0
- Outer 56-case test set stays untouched until Stage 8

**What changed vs. the previous notebook:**
- All artifacts now go under a clearly named, organized directory and are zipped for persistence
- `medpy` is completely removed — Dice/IoU/HD95 computed with scipy only (no dependency conflict)
- HD95 is computed in physical mm using NIfTI voxel spacing, not raw voxel counts
- Training cell checks for an existing checkpoint first, so re-running the notebook never accidentally retrains
- Explicit verification cells after training and after inference, before anything else proceeds

**How to run:** Run cells top to bottom, in order, **in one single interactive session**. Training, checkpoint packaging, inference, prediction packaging, metrics, and the final ZIP all happen in this same run — there is no second expensive step and no required re-commit afterward. Stop and read the printed output at Stage 7 and Stage 10 before continuing — those are checkpoints, not just narration. See Stage 13 for exactly how to persist the result without retraining.

## STAGE 1 — Environment & Path Setup

In [1]:
import os
import json
import shutil
import random
import datetime
import numpy as np
import pandas as pd
import nibabel as nib

# ---- Source dataset (existing, verified) ----
DATASET_PATH = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"

# ---- Working directories (organized under one named folder) ----
WORK_DIR = "/kaggle/working/neuroshield_track2"
NNUNET_RAW = f"{WORK_DIR}/nnUNet_raw"
NNUNET_PREPROCESSED = f"{WORK_DIR}/nnUNet_preprocessed"
NNUNET_RESULTS = f"{WORK_DIR}/nnUNet_results"
TEST_IMAGES = f"{WORK_DIR}/imagesTs"
TEST_PREDICTIONS = f"{WORK_DIR}/test_predictions"

# ---- Final, clearly-named persistent artifact directory ----
FINAL_ARTIFACT_DIR = "/kaggle/working/neuroshield_track2_model_a"
ARTIFACT_MODEL_DIR = f"{FINAL_ARTIFACT_DIR}/model"
ARTIFACT_PRED_DIR = f"{FINAL_ARTIFACT_DIR}/test_predictions"
ARTIFACT_METRICS_DIR = f"{FINAL_ARTIFACT_DIR}/metrics"
ARTIFACT_SPLITS_DIR = f"{FINAL_ARTIFACT_DIR}/splits"

DATASET_NAME = "Dataset001_BraTS2020"
DATASET_ID = "001"
TRAINER = "nnUNetTrainer_100epochs"
CONFIG = "2d"
FOLD = "0"
SEED = 42

for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, TEST_IMAGES, TEST_PREDICTIONS,
          ARTIFACT_MODEL_DIR, ARTIFACT_PRED_DIR, ARTIFACT_METRICS_DIR, ARTIFACT_SPLITS_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("Dataset path exists:", os.path.exists(DATASET_PATH))
print("nnUNet_raw          :", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed :", os.environ["nnUNet_preprocessed"])
print("nnUNet_results      :", os.environ["nnUNet_results"])
print("Final artifact dir  :", FINAL_ARTIFACT_DIR)

Dataset path exists: True
nnUNet_raw          : /kaggle/working/neuroshield_track2/nnUNet_raw
nnUNet_preprocessed : /kaggle/working/neuroshield_track2/nnUNet_preprocessed
nnUNet_results      : /kaggle/working/neuroshield_track2/nnUNet_results
Final artifact dir  : /kaggle/working/neuroshield_track2_model_a


In [2]:
!pip install -q nnunetv2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 9.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the

## STAGE 2 — Dataset Discovery (unchanged from prior notebook)

In [3]:
TRAIN_DIR = os.path.join(DATASET_PATH, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")

cases = []
for entry in sorted(os.listdir(TRAIN_DIR)):
    case_path = os.path.join(TRAIN_DIR, entry)
    if not os.path.isdir(case_path):
        continue
    files = os.listdir(case_path)
    def find(suffix, files=files):
        matches = [f for f in files if f.lower().endswith(suffix)]
        return os.path.join(case_path, matches[0]) if matches else None

    row = {
        "case_id": entry,
        "flair": find("flair.nii"),
        "t1": find("t1.nii"),
        "t1ce": find("t1ce.nii"),
        "t2": find("t2.nii"),
        "seg": find("seg.nii"),
    }
    if all(row.values()):
        cases.append(row)

print(f"Usable cases with all 4 modalities + mask: {len(cases)}")
assert len(cases) == 368, f"Expected 368 usable cases, found {len(cases)} — dataset mount may differ."

Usable cases with all 4 modalities + mask: 368


## STAGE 3 — Fixed Outer Split (seed=42, 257/55/56 — unchanged)

In [4]:
random.seed(SEED)

case_ids = [c["case_id"] for c in cases]
random.shuffle(case_ids)

n = len(case_ids)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_ids = set(case_ids[:n_train])
val_ids = set(case_ids[n_train:n_train + n_val])
test_ids = set(case_ids[n_train + n_val:])

print(f"Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")
assert len(train_ids) == 257 and len(val_ids) == 55 and len(test_ids) == 56, \
    "Split counts do not match the established 257/55/56 outer split."

assert not (train_ids & val_ids)
assert not (val_ids & test_ids)
assert not (train_ids & test_ids)
print("No overlap between train/val/test — confirmed.")

Train: 257  Val: 55  Test: 56
No overlap between train/val/test — confirmed.


In [5]:
df = pd.DataFrame(cases)

def assign_split(cid):
    if cid in train_ids: return "train"
    if cid in val_ids: return "val"
    return "test"

df["split"] = df["case_id"].apply(assign_split)

manifest_path = os.path.join(ARTIFACT_SPLITS_DIR, "braintumor_manifest.csv")
df.to_csv(manifest_path, index=False)

print(df["split"].value_counts())
print("Manifest saved to:", manifest_path)

split
train    257
test      56
val       55
Name: count, dtype: int64
Manifest saved to: /kaggle/working/neuroshield_track2_model_a/splits/braintumor_manifest.csv


## STAGE 4 — nnU-Net Dataset Preparation (conversion + dataset.json)

In [6]:
RAW_BASE = os.path.join(NNUNET_RAW, DATASET_NAME)
imagesTr = os.path.join(RAW_BASE, "imagesTr")
labelsTr = os.path.join(RAW_BASE, "labelsTr")
os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

trainval = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
test_cases = df[df["split"] == "test"].reset_index(drop=True)
print(f"Cases going into nnU-Net's training pool: {len(trainval)}")
print(f"Outer test cases (untouched until Stage 8): {len(test_cases)}")
assert len(test_cases) == 56, "Outer test set must contain exactly 56 cases."

Cases going into nnU-Net's training pool: 312
Outer test cases (untouched until Stage 8): 56


In [7]:
modality_map = {"flair": "0000", "t1": "0001", "t1ce": "0002", "t2": "0003"}

# Skip conversion if already done (avoids redundant compute on re-run)
already_converted = os.path.exists(imagesTr) and len(os.listdir(imagesTr)) >= len(trainval) * 4

if already_converted:
    print("imagesTr/labelsTr already populated — skipping conversion.")
else:
    for i, row in trainval.iterrows():
        case_id = row["case_id"]
        for mod, channel in modality_map.items():
            img = nib.load(row[mod])
            nib.save(img, os.path.join(imagesTr, f"{case_id}_{channel}.nii.gz"))

        seg_img = nib.load(row["seg"])
        seg_data = seg_img.get_fdata()
        seg_data[seg_data == 4] = 3  # nnU-Net needs consecutive labels: 0,1,2,3
        remapped = nib.Nifti1Image(seg_data.astype(np.uint8), seg_img.affine, seg_img.header)
        nib.save(remapped, os.path.join(labelsTr, f"{case_id}.nii.gz"))

        if (i + 1) % 50 == 0:
            print(f"Converted {i+1}/{len(trainval)} cases")
    print("Conversion complete.")

Converted 50/312 cases
Converted 100/312 cases
Converted 150/312 cases
Converted 200/312 cases
Converted 250/312 cases
Converted 300/312 cases
Conversion complete.


In [8]:
# Verify one converted case
sample_id = trainval.iloc[0]["case_id"]
print("Image files:", [f for f in sorted(os.listdir(imagesTr)) if f.startswith(sample_id)])
sample_label = nib.load(os.path.join(labelsTr, f"{sample_id}.nii.gz")).get_fdata()
print("Label values after remap:", np.unique(sample_label))  # expect [0. 1. 2. 3.]

Image files: ['BraTS20_Training_001_0000.nii.gz', 'BraTS20_Training_001_0001.nii.gz', 'BraTS20_Training_001_0002.nii.gz', 'BraTS20_Training_001_0003.nii.gz']
Label values after remap: [0. 1. 2. 3.]


In [9]:
dataset_json = {
    "channel_names": {"0": "FLAIR", "1": "T1", "2": "T1CE", "3": "T2"},
    "labels": {
        "background": 0,
        "necrotic_non_enhancing_core": 1,
        "edema": 2,
        "enhancing_tumor": 3
    },
    "numTraining": len(trainval),
    "file_ending": ".nii.gz"
}

with open(os.path.join(RAW_BASE, "dataset.json"), "w") as f:
    json.dump(dataset_json, f, indent=4)

print(json.dumps(dataset_json, indent=4))

{
    "channel_names": {
        "0": "FLAIR",
        "1": "T1",
        "2": "T1CE",
        "3": "T2"
    },
    "labels": {
        "background": 0,
        "necrotic_non_enhancing_core": 1,
        "edema": 2,
        "enhancing_tumor": 3
    },
    "numTraining": 312,
    "file_ending": ".nii.gz"
}


## STAGE 5 — Preprocessing

In [10]:
# Skip if already preprocessed (avoids redundant compute on re-run)
preprocessed_marker = os.path.join(NNUNET_PREPROCESSED, DATASET_NAME)
if os.path.exists(preprocessed_marker) and len(os.listdir(preprocessed_marker)) > 0:
    print("Preprocessed data already exists — skipping plan_and_preprocess.")
else:
    !nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity

Fingerprint extraction...
Dataset001_BraTS2020
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Extracting dataset fingerprint: 100%|█████████| 312/312 [01:56<00:00,  2.68it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [139. 169. 138.], 3d_lowres: [139, 169, 138]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreproces

In [11]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is NOT available")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [12]:
import os
import glob

print("=== SPLIT / MODEL SAFETY CHECK ===")

for root in [
    "/kaggle/working/neuroshield_track2",
    "/kaggle/working/neuroshield_track2_model_a",
]:
    print(f"\n{root}")
    if os.path.exists(root):
        for path in glob.glob(root + "/**/*", recursive=True):
            if os.path.isfile(path) and (
                "manifest" in os.path.basename(path).lower()
                or "checkpoint" in os.path.basename(path).lower()
            ):
                print(path)
    else:
        print("Directory does not exist")

print("\n=== CHECK COMPLETE ===")

=== SPLIT / MODEL SAFETY CHECK ===

/kaggle/working/neuroshield_track2

/kaggle/working/neuroshield_track2_model_a
/kaggle/working/neuroshield_track2_model_a/splits/braintumor_manifest.csv

=== CHECK COMPLETE ===


In [13]:
import os
import pandas as pd

manifest = "/kaggle/working/neuroshield_track2_model_a/splits/braintumor_manifest.csv"

df = pd.read_csv(manifest)

print("=== FINAL PRE-TRAINING CHECK ===")
print("Manifest:", manifest)
print("Train:", (df["split"] == "train").sum())
print("Val:", (df["split"] == "val").sum())
print("Test:", (df["split"] == "test").sum())

# Safety check: existing trained checkpoint
results_dir = "/kaggle/working/neuroshield_track2/nnUNet_results"

checkpoints = []
if os.path.exists(results_dir):
    for root, dirs, files in os.walk(results_dir):
        for f in files:
            if f in ["checkpoint_final.pth", "checkpoint_best.pth"]:
                checkpoints.append(os.path.join(root, f))

print("\nExisting checkpoints:")
if checkpoints:
    for p in checkpoints:
        print("FOUND:", p)
else:
    print("None found")

assert (df["split"] == "train").sum() == 257
assert (df["split"] == "val").sum() == 55
assert (df["split"] == "test").sum() == 56

print("\n=== SAFE TO PROCEED ===")

=== FINAL PRE-TRAINING CHECK ===
Manifest: /kaggle/working/neuroshield_track2_model_a/splits/braintumor_manifest.csv
Train: 257
Val: 55
Test: 56

Existing checkpoints:
None found

=== SAFE TO PROCEED ===


## STAGE 6 — Model A Training
**Safety check built in:** this cell checks for an existing final checkpoint before training. If found, training is skipped entirely — re-running this notebook will never silently retrain and burn GPU hours. Delete the checkpoint file manually if you deliberately want to retrain.

In [14]:
expected_ckpt_dir = os.path.join(
    NNUNET_RESULTS, DATASET_NAME,
    f"{TRAINER}__nnUNetPlans__{CONFIG}", f"fold_{FOLD}"
)
final_ckpt = os.path.join(expected_ckpt_dir, "checkpoint_final.pth")

if os.path.exists(final_ckpt):
    print("Final checkpoint already exists — SKIPPING training to avoid accidental retrain:")
    print(final_ckpt)
else:
    print("No final checkpoint found. Starting/resuming training.")
    print("Note: '--c' is safe to use even in a fresh environment — nnU-Net will simply report")
    print("'no checkpoint available, starting new training' and begin cleanly. It only resumes")
    print("from a checkpoint that matches this exact dataset/trainer/fold, never an unrelated one.")
    !nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} -tr {TRAINER} --c

No final checkpoint found. Starting/resuming training.
Note: '--c' is safe to use even in a fresh environment — nnU-Net will simply report
'no checkpoint available, starting new training' and begin cleanly. It only resumes
from a checkpoint that matches this exact dataset/trainer/fold, never an unrelated one.

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
##########################################################

## STAGE 7 — CHECKPOINT VERIFICATION (stop and check before continuing)


In [15]:
best_ckpt = os.path.join(expected_ckpt_dir, "checkpoint_best.pth")
plans_json = os.path.join(expected_ckpt_dir, "plans.json")
ds_json = os.path.join(expected_ckpt_dir, "dataset.json")

required_files = {
    "checkpoint_final.pth": final_ckpt,
    "checkpoint_best.pth": best_ckpt,
    "plans.json": plans_json,
    "dataset.json": ds_json,
}

all_ok = True
for name, path in required_files.items():
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"[{status}] {name:<22} {size_mb:8.2f} MB   {path}")
    if not exists:
        all_ok = False

assert all_ok, "One or more required checkpoint/config files are missing. Do not proceed to Stage 8."
print("\nAll required checkpoint files verified. Safe to proceed to Stage 8.")

[OK] checkpoint_final.pth     165.14 MB   /kaggle/working/neuroshield_track2/nnUNet_results/Dataset001_BraTS2020/nnUNetTrainer_100epochs__nnUNetPlans__2d/fold_0/checkpoint_final.pth
[OK] checkpoint_best.pth      165.14 MB   /kaggle/working/neuroshield_track2/nnUNet_results/Dataset001_BraTS2020/nnUNetTrainer_100epochs__nnUNetPlans__2d/fold_0/checkpoint_best.pth
[MISSING] plans.json                 0.00 MB   /kaggle/working/neuroshield_track2/nnUNet_results/Dataset001_BraTS2020/nnUNetTrainer_100epochs__nnUNetPlans__2d/fold_0/plans.json
[MISSING] dataset.json               0.00 MB   /kaggle/working/neuroshield_track2/nnUNet_results/Dataset001_BraTS2020/nnUNetTrainer_100epochs__nnUNetPlans__2d/fold_0/dataset.json


AssertionError: One or more required checkpoint/config files are missing. Do not proceed to Stage 8.

## STAGE 8 — BACKUP / PERSIST MODEL
This runs immediately after training in the **same session** — no separate commit/re-run needed. The saved structure preserves nnU-Net's own required layout (`model/Dataset001_BraTS2020/<trainer>__nnUNetPlans__<config>/fold_X/`), so it can be pointed to directly as `nnUNet_results` in a future session for inference, with no reshaping needed.

In [ ]:
# Copy the entire trained fold directory, preserving nnU-Net's required nested structure:
# model/Dataset001_BraTS2020/<trainer>__nnUNetPlans__<config>/fold_X/
artifact_dataset_dir = os.path.join(ARTIFACT_MODEL_DIR, DATASET_NAME)
artifact_config_dir = os.path.join(artifact_dataset_dir, f"{TRAINER}__nnUNetPlans__{CONFIG}")
artifact_fold_dir = os.path.join(artifact_config_dir, f"fold_{FOLD}")
os.makedirs(artifact_config_dir, exist_ok=True)

if os.path.exists(artifact_fold_dir):
    shutil.rmtree(artifact_fold_dir)
shutil.copytree(expected_ckpt_dir, artifact_fold_dir)

# Also copy the dataset.json used for raw conversion (channel/label meaning) at the dataset level
shutil.copy(os.path.join(RAW_BASE, "dataset.json"), os.path.join(artifact_dataset_dir, "dataset.json"))

print("Model artifact copied to:", artifact_fold_dir)
print("Full nnU-Net-compatible structure:")
for root, dirs, files in os.walk(ARTIFACT_MODEL_DIR):
    level = root.replace(ARTIFACT_MODEL_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files:
        print("  " * (level + 1) + f)

In [ ]:
training_metadata = {
    "project": "NeuroShield",
    "track": "Track 2 - Brain Tumor Segmentation",
    "dataset": "BraTS2020",
    "total_usable_cases": len(cases),
    "train_count": len(train_ids),
    "val_count": len(val_ids),
    "test_count": len(test_ids),
    "seed": SEED,
    "model": "2D nnU-Net (Model A)",
    "trainer": TRAINER,
    "configuration": CONFIG,
    "fold": FOLD,
    "epochs": 100,
    "run_datetime_utc": datetime.datetime.utcnow().isoformat()
}

with open(os.path.join(FINAL_ARTIFACT_DIR, "training_metadata.json"), "w") as f:
    json.dump(training_metadata, f, indent=4)

print(json.dumps(training_metadata, indent=4))